<a href="https://colab.research.google.com/github/thakrenikhil/basic_rag/blob/main/Basic_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 58.4 MB/s eta 0:00:00


In [27]:
# !pip install sentence-transformers faiss-cpu google-generativeai torch numpy

"""
Splits raw documents into overlapping text chunks small enough to embed
and retrieve meaningfully. Overlap keeps context from getting cut off
mid-idea at chunk boundaries.
"""

import re
from dataclasses import dataclass


@dataclass
class Chunk:
    text: str
    doc_id: str
    chunk_id: int
    start_char: int


def _split_sentences(text: str) -> list[str]:
    # Lightweight sentence splitter -- good enough for chunking, not
    # meant to be a real NLP sentence tokenizer.
    text = re.sub(r"\s+", " ", text).strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s for s in sentences if s]


def chunk_document(
    text: str,
    doc_id: str,
    chunk_size: int = 500,
    overlap: int = 100,
) -> list[Chunk]:
    """
    Groups sentences into chunks up to `chunk_size` characters, carrying
    the last `overlap` characters of each chunk into the next one so
    retrieval doesn't lose context that straddles a boundary.
    """
    sentences = _split_sentences(text)
    chunks: list[Chunk] = []

    current = ""
    start_char = 0
    chunk_id = 0

    for sentence in sentences:
        if len(current) + len(sentence) + 1 > chunk_size and current:
            chunks.append(Chunk(current.strip(), doc_id, chunk_id, start_char))
            chunk_id += 1
            # carry the tail of the previous chunk forward as overlap
            tail = current[-overlap:] if overlap > 0 else ""
            start_char += len(current) - len(tail)
            current = tail + " " + sentence
        else:
            current = f"{current} {sentence}".strip()

    if current.strip():
        chunks.append(Chunk(current.strip(), doc_id, chunk_id, start_char))

    return chunks


def chunk_documents(documents: dict[str, str], chunk_size: int = 500, overlap: int = 100) -> list[Chunk]:
    """
    documents: {doc_id: full_text}
    Returns a flat list of Chunk objects across all documents.
    """
    all_chunks: list[Chunk] = []
    for doc_id, text in documents.items():
        all_chunks.extend(chunk_document(text, doc_id, chunk_size, overlap))
    return all_chunks

In [ ]:
"""
Wraps a sentence-transformers model so the rest of the pipeline doesn't
need to know which model or device is in use.
"""

import numpy as np
from sentence_transformers import SentenceTransformer


class Embedder:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2", device: str | None = None):
        # device=None lets sentence-transformers auto-pick cuda/mps/cpu
        self.model = SentenceTransformer(model_name, device=device)
        self.dim = self.model.get_sentence_embedding_dimension()

    def encode(self, texts: list[str], batch_size: int = 32, normalize: bool = True) -> np.ndarray:
        """
        Returns a (len(texts), dim) float32 array.
        normalize=True makes cosine similarity equivalent to inner product,
        which matters if you later switch the FAISS index to IndexFlatIP.
        """
        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=len(texts) > 50,
            convert_to_numpy=True,
            normalize_embeddings=normalize,
        )
        return embeddings.astype("float32")

In [28]:
"""
Thin wrapper around a FAISS index plus a parallel metadata list, since
FAISS itself only stores vectors and integer ids -- it knows nothing
about your chunk text or which document it came from.
"""

import pickle
from pathlib import Path

import faiss
import numpy as np

# from chunking import Chunk


class VectorStore:
    def __init__(self, dim: int):
        # Flat = exact brute-force search. Correct and simple; swap for
        # IndexIVFFlat or IndexHNSWFlat later if you need to scale past
        # ~100k-1M vectors and exact search gets too slow.
        self.index = faiss.IndexFlatL2(dim)
        self.metadata: list[Chunk] = []

    def add(self, embeddings: np.ndarray, chunks: list[Chunk]) -> None:
        assert embeddings.shape[0] == len(chunks), "embeddings/chunks count mismatch"
        self.index.add(embeddings)
        self.metadata.extend(chunks)

    def search(self, query_embedding: np.ndarray, top_k: int = 5) -> list[tuple[Chunk, float]]:
        """
        query_embedding: shape (1, dim)
        Returns [(chunk, distance), ...] sorted by ascending distance
        (lower = more similar, since this is L2).
        """
        distances, indices = self.index.search(query_embedding, top_k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue  # FAISS pads with -1 if fewer than top_k results exist
            results.append((self.metadata[idx], float(dist)))
        return results

    def save(self, dir_path: str) -> None:
        path = Path(dir_path)
        path.mkdir(parents=True, exist_ok=True)
        faiss.write_index(self.index, str(path / "index.faiss"))
        with open(path / "metadata.pkl", "wb") as f:
            pickle.dump(self.metadata, f)

    @classmethod
    def load(cls, dir_path: str) -> "VectorStore":
        path = Path(dir_path)
        index = faiss.read_index(str(path / "index.faiss"))
        store = cls(index.d)
        store.index = index
        with open(path / "metadata.pkl", "rb") as f:
            store.metadata = pickle.load(f)
        return store

In [32]:
# @title
"""
Wraps the Gemini API call for the final answer-generation step.
Requires GEMINI_API_KEY to be set in the environment.
"""

import os

import google.generativeai as genai

# from chunking import Chunk

PROMPT_TEMPLATE = """You are answering a question using only the provided context.
If the context doesn't contain the answer, say you don't know -- do not make something up.

Context:
{context}

Question: {question}

Answer:"""


class GeminiClient:
    def __init__(self, model_name: str = "gemini-2.0-flash", api_key: str | None = None):
        api_key = api_key or os.environ.get("GEMINI_API_KEY")
        if not api_key:
            raise ValueError("Set GEMINI_API_KEY in your environment or pass api_key explicitly.")
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel("gemini-3.5-flash")

    def build_prompt(self, question: str, chunks: list[Chunk]) -> str:
        context = "\n\n---\n\n".join(
            f"[{c.doc_id} #{c.chunk_id}] {c.text}" for c in chunks
        )
        return PROMPT_TEMPLATE.format(context=context, question=question)

    def answer(self, question: str, chunks: list[Chunk]) -> str:
        prompt = self.build_prompt(question, chunks)
        response = self.model.generate_content(prompt)
        return response.text

In [33]:
"""
Ties the whole flow together:

documents -> chunk_documents() -> embedder.encode() -> FAISS index
query -> query embedding -> FAISS.search() -> top_k chunks
-> prompt construction -> Gemini -> answer
"""

# from chunking import chunk_documents
# from embedder import Embedder
# from vector_store import VectorStore
# from gemini_client import GeminiClient


class RAGPipeline:
    def __init__(self, embed_model: str = "all-MiniLM-L6-v2", gemini_model: str = "gemini-2.0-flash"):
        self.embedder = Embedder(embed_model)
        self.store = VectorStore(self.embedder.dim)
        self.gemini = GeminiClient(gemini_model)

    def index(self, documents: dict[str, str], chunk_size: int = 500, overlap: int = 100) -> None:
        """documents: {doc_id: full_text}"""
        chunks = chunk_documents(documents, chunk_size, overlap)
        embeddings = self.embedder.encode([c.text for c in chunks])
        self.store.add(embeddings, chunks)
        print(f"Indexed {len(chunks)} chunks from {len(documents)} document(s).")

    def query(self, question: str, top_k: int = 5) -> str:
        query_embedding = self.embedder.encode([question])
        top_chunks_with_scores = self.store.search(query_embedding, top_k)
        top_chunks = [c for c, _ in top_chunks_with_scores]

        if not top_chunks:
            return "No indexed documents to search yet."

        return self.gemini.answer(question, top_chunks)

    def save(self, dir_path: str = "rag_index") -> None:
        self.store.save(dir_path)

    def load(self, dir_path: str = "rag_index") -> None:
        self.store = VectorStore.load(dir_path)


if __name__ == "__main__":
    pipeline = RAGPipeline()

    docs = {
        "doc1": "PyTorch tensors are the core data structure, similar to NumPy arrays but with GPU support and autograd tracking for gradients.",
        "doc2": "FAISS is a library for efficient similarity search over dense vectors, developed by Meta AI research.",
        "doc3": "I am Daemon Targerion, protector of the Realm, King of the Andals"
    }

    pipeline.index(docs)
    answer = pipeline.query("Who is FAISS?")
    print(answer)
    # print(metadata)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_1508/3941164017.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()


Indexed 3 chunks from 3 document(s).
Based on the provided context, FAISS is a library for efficient similarity search over dense vectors, developed by Meta AI research.
